In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
print("Hi")
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
# device = torch.device('cpu')

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # (32, 3, 128, 128) -> (32, 8, 64, 64) -> (32, 16, 32, 32) -> (32, 32, 16, 16)
        # -> (32, 64, 8, 8)
        self.conv1 = nn.Conv2d(3, 8, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(8)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(8, 16, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(16)
        self.conv3 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 6)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)
model.load_state_dict(torch.load('my_custom_cnn_fruits.pth', map_location=device))
model.eval()

test_transform = A.Compose([
    A.Resize(128, 128),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), 
    ToTensorV2()
])

print("Complete")

Hi
Complete


In [15]:
import os
import shutil
from PIL import Image
import numpy as np
import cv2
# 폴더를 생성할 필요 없이, 얼마나 틀리는지 봐야겠어.

wrong_dirs = "images/kaggle_data/dataset/wrong_images"

os.makedirs(wrong_dirs, exist_ok=True)

kinds = []



total_count = 0
correct_count = 0
wrong_count = 0
fruits = ["apple", "banana", "orange"]

for fruit in fruits:
    kinds.append(f"fresh_{fruit}")
    kinds.append(f"rotten_{fruit}")

for fruit in fruits:
    path = f"images/kaggle_data/dataset/{fruit}"
    categories = [f"fresh_{fruit}", f"rotten_{fruit}"]
    for category in categories:
        image_path = os.path.join(path, category)
        for image in os.listdir(image_path):
            if image.endswith(('jpg', 'jpeg', 'png')):
                total_count += 1
                if total_count % 1000 == 0:
                    print(f"total_count: {total_count}")
                full_path = os.path.join(image_path, image)
                
                # pil_image = Image.open(full_path).convert('RGB')
                # image_np = np.array(pil_image)
                # image = cv2.cvtColor(image_np, cv2.COLOR_BGR2RGB)

                pil_image = Image.open(full_path).convert('RGB')
                image_np = np.array(pil_image)

                
                aug = test_transform(image=image_np)
                input_tensor = aug['image'].unsqueeze(0)

                with torch.no_grad():
                    outputs = model(input_tensor.to(device))
                    pred = outputs.argmax(1)

                pred_text = kinds[pred]

                if pred_text == category:
                    correct_count += 1
                elif pred_text != category:
                    wrong_count += 1
                    shutil.copy(os.path.join(image_path, image), os.path.join(wrong_dirs, image))

print(f"Total: {total_count}, Correct: {correct_count}, Wrong: {wrong_count}")

total_count: 1000
total_count: 2000
total_count: 3000
total_count: 4000
total_count: 5000
total_count: 6000
total_count: 7000
total_count: 8000
total_count: 9000
total_count: 10000
total_count: 11000
total_count: 12000
total_count: 13000
total_count: 14000
total_count: 15000
total_count: 16000
Total: 16831, Correct: 16474, Wrong: 357
